# 🎯 Watermark Detection YOLO Training - Kaggle

## Smart Auto-Labeling Strategy:
✅ **Use your manual coordinates** to automatically label all frames!
✅ **No manual annotation needed** - we expand your known regions slightly
✅ **Accurate labels** - based on where you know watermarks actually appear

## Kaggle-Specific Setup:
- 📁 Upload videos as **Kaggle Dataset** (easier than file upload)
- 🚀 Enable **GPU** in notebook settings (right sidebar → Accelerator → GPU T4 x2)
- 💾 Auto-saves outputs to `/kaggle/working/` (downloadable)

## Workflow:
1. Upload videos as Kaggle Dataset
2. Extract & crop frames
3. Auto-label using manual coordinates
4. Train YOLO model
5. Download best.pt from Output section

⏰ **Total time: 2-4 hours** (fully automated on Kaggle GPU!)

---

## ⚙️ IMPORTANT: Enable GPU!

**Before running any cells:**

1. Click **⋮** (three dots) on the right sidebar
2. Select **Notebook Settings** or **Accelerator**
3. Choose: **GPU T4 x2** (free tier)
4. Click **Save**

✅ GPU is required for training!

## Step 1: Install YOLO

In [ ]:
# Install ultralytics (YOLO)
!pip install ultralytics==8.0.210 -q

import torch
print(f"✅ Installation complete!")
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected! Enable GPU in Settings → Accelerator → GPU T4 x2")

## Step 2: Setup Directories & Check Dataset

**How to upload videos to Kaggle:**

### Method 1: Create Kaggle Dataset (Recommended)
1. Go to https://www.kaggle.com/datasets
2. Click **New Dataset**
3. Upload your videos (.mp4 files)
4. Make it **Private**
5. Click **Create**
6. In THIS notebook: Click **+ Add Data** (right sidebar) → Search your dataset → Add

### Method 2: Direct Upload (For testing)
- Upload videos directly using the file browser on the right sidebar
- Note: Files will be lost when notebook restarts

**After adding dataset, update the path below!**

In [ ]:
import os
import cv2
import numpy as np
import shutil

# ========================================
# 🎯 AUTO-DETECT VIDEO SOURCE
# ========================================
# This will automatically find videos from either:
# 1. GitHub Actions upload: /kaggle/input/USERNAME/watermark-training-videos/
# 2. Manual dataset: /kaggle/input/your-dataset-name/
# 3. Direct upload: /kaggle/working/videos/

# Try common dataset paths
possible_paths = [
    '/kaggle/input/watermark-training-videos/',  # GitHub Actions default
]

# Add all datasets in /kaggle/input/
if os.path.exists('/kaggle/input/'):
    for item in os.listdir('/kaggle/input/'):
        item_path = f'/kaggle/input/{item}/'
        if os.path.isdir(item_path):
            possible_paths.append(item_path)

# Add working directory option
possible_paths.append('/kaggle/working/videos/')

VIDEO_SOURCE = None

print("📁 Searching for video files...\n")

for path in possible_paths:
    if os.path.exists(path):
        # Check if it contains .mp4 files (directly or in subfolders)
        video_files = []
        for root, dirs, files in os.walk(path):
            video_files.extend([f for f in files if f.endswith('.mp4')])
        
        if video_files:
            VIDEO_SOURCE = path
            print(f"✅ Found video dataset: {path}")
            print(f"📹 Found {len(video_files)} video files:")
            for vf in video_files[:10]:
                print(f"   - {vf}")
            if len(video_files) > 10:
                print(f"   ... and {len(video_files) - 10} more")
            break

if VIDEO_SOURCE is None:
    print("❌ No video files found!")
    print("\n💡 Make sure you:")
    print("1. Uploaded videos as Kaggle Dataset, OR")
    print("2. Added dataset via '+ Add Data' button, OR")
    print("3. Uploaded videos to /kaggle/working/videos/")
    print("\n🔧 If using GitHub Actions, check the workflow uploaded correctly")
    raise FileNotFoundError("No video files found in any expected location")

# Create output directories in /kaggle/working/ (Kaggle's writable directory)
os.makedirs('/kaggle/working/dataset/images', exist_ok=True)
os.makedirs('/kaggle/working/dataset/labels', exist_ok=True)

print(f"\n✅ Output directories created in /kaggle/working/")

## Step 3: Extract & Crop Frames

This will:
1. Extract every 5th frame from videos (faster extraction)
2. Detect embedded video area (black border detection)
3. Crop to just the embedded video
4. Save cropped frames

**Note:** Adjust `frame_interval` and `max_frames_per_video` based on your needs!

In [ ]:
# Extract frames from videos AND crop to embedded video area

output_folder = '/kaggle/working/dataset/images'
frame_interval = 5  # Extract every 5th frame (faster than 30)
max_frames_per_video = 3000  # Max frames per video

def detect_video_area(frame):
    """
    Detects the embedded video area by scanning from center outward.
    Same logic as your video_editing.py
    """
    height, width = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    black_thresh = 10
    black_ratio_thresh = 0.95
    
    center_x = width // 2
    center_y = height // 2
    
    # Find top boundary
    for y in range(center_y, 0, -1):
        row = gray[y, :]
        if np.mean(row < black_thresh) > black_ratio_thresh:
            start_y = y + 1
            break
    else:
        start_y = 0
    
    # Find bottom boundary
    for y in range(center_y, height):
        row = gray[y, :]
        if np.mean(row < black_thresh) > black_ratio_thresh:
            end_y = y - 1
            break
    else:
        end_y = height - 1
    
    # Find left boundary
    for x in range(center_x, 0, -1):
        col = gray[:, x]
        if np.mean(col < black_thresh) > black_ratio_thresh:
            start_x = x + 1
            break
    else:
        start_x = 0
    
    # Find right boundary
    for x in range(center_x, width):
        col = gray[:, x]
        if np.mean(col < black_thresh) > black_ratio_thresh:
            end_x = x - 1
            break
    else:
        end_x = width - 1
    
    box_width = end_x - start_x
    box_height = end_y - start_y
    
    if box_width <= 0 or box_height <= 0:
        return None
    
    crop_margin = 5
    return (start_x + crop_margin, start_y + crop_margin, 
            box_width - crop_margin, box_height - crop_margin)

print("🎬 Extracting and cropping frames...")
frame_count = 0

# Get all video files from source (including subdirectories)
video_files = []
for root, dirs, files in os.walk(VIDEO_SOURCE):
    for file in files:
        if file.endswith('.mp4'):
            video_files.append(os.path.join(root, file))

if not video_files:
    print(f"❌ No .mp4 files found in {VIDEO_SOURCE}")
    print("Please check Step 2")
    raise FileNotFoundError("No video files to process")
else:
    print(f"📹 Processing {len(video_files)} videos...\n")
    
    for video_path in video_files:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        
        print(f"Processing: {os.path.basename(video_path)}")
        
        cap = cv2.VideoCapture(video_path)
        
        # Detect video area from first frame
        ret, first_frame = cap.read()
        if not ret:
            print(f"  ❌ Could not read {video_path}")
            continue
        
        video_area = detect_video_area(first_frame)
        if not video_area:
            print(f"  ⚠️ Could not detect video area, using full frame")
            h, w = first_frame.shape[:2]
            video_area = (0, 0, w, h)
        
        x, y, w, h = video_area
        print(f"  📐 Crop area: {w}x{h} at ({x}, {y})")
        
        # Reset to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        count = 0
        extracted = 0
        
        while cap.isOpened() and extracted < max_frames_per_video:
            ret, frame = cap.read()
            if not ret:
                break
            
            if count % frame_interval == 0:
                # CROP to embedded video area
                cropped_frame = frame[y:y+h, x:x+w]
                
                output_path = os.path.join(output_folder, f"{video_name}_frame_{count:05d}.jpg")
                cv2.imwrite(output_path, cropped_frame)
                extracted += 1
                frame_count += 1
            
            count += 1
        
        cap.release()
        print(f"  ✅ Extracted {extracted} cropped frames")
    
    print(f"\n🎉 Total extracted: {frame_count} CROPPED frames")
    print(f"📍 Frames saved to: {output_folder}")

## Step 4: Auto-Label Using Your Manual Coordinates

Configure your watermark regions below and auto-generate labels!

In [ ]:
# ========================================
# 🎯 CONFIGURE YOUR WATERMARK REGIONS HERE
# ========================================
# These are normalized coordinates (0.0 to 1.0)
# Update these to match your video_editing.py settings

manual_regions = [
    {'x': 0.5, 'y': 1.0, 'width': 0.15, 'height': 0.70},  # Example: bottom-center
]

expansion_factor = 1.1  # 10% larger for better coverage

print(f"📍 Using {len(manual_regions)} watermark regions")
print(f"📏 Expanding regions by {(expansion_factor-1)*100:.0f}%\n")

# Auto-label all frames
images_folder = '/kaggle/working/dataset/images'
labels_folder = '/kaggle/working/dataset/labels'

labeled_count = 0
total_labels = 0

image_files = [f for f in os.listdir(images_folder) if f.endswith('.jpg')]

for img_file in image_files:
    img_path = os.path.join(images_folder, img_file)
    frame = cv2.imread(img_path)
    
    if frame is None:
        continue
    
    frame_h, frame_w = frame.shape[:2]
    
    # Create label file
    label_file = os.path.join(labels_folder, img_file.replace('.jpg', '.txt'))
    
    with open(label_file, 'w') as f:
        for region in manual_regions:
            x_norm = region['x']
            y_norm = region['y']
            w_norm = region['width']
            h_norm = region['height']
            
            # Expand the region
            w_expanded = w_norm * expansion_factor
            h_expanded = h_norm * expansion_factor
            
            # Write YOLO format label
            f.write(f"0 {x_norm:.6f} {y_norm:.6f} {w_expanded:.6f} {h_expanded:.6f}\n")
            total_labels += 1
    
    labeled_count += 1
    
    if labeled_count % 1000 == 0:
        print(f"Labeled {labeled_count} images...")

print(f"\n✅ Auto-labeled {labeled_count} images with {total_labels} watermark regions!")
print(f"   Average: {total_labels/labeled_count:.1f} watermarks per frame")

## Step 4.1: Visualize Labeled Data 👁️

Preview labeled frames to verify boxes are correct!

In [ ]:
import matplotlib.pyplot as plt
import random

def draw_yolo_boxes(image_path, label_path):
    """Draw YOLO format bounding boxes on image"""
    img = cv2.imread(image_path)
    img_h, img_w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id, x_center, y_center, width, height = map(float, parts)
                
                # Convert to pixel coordinates
                x_center_px = int(x_center * img_w)
                y_center_px = int(y_center * img_h)
                box_w = int(width * img_w)
                box_h = int(height * img_h)
                
                x1 = int(x_center_px - box_w / 2)
                y1 = int(y_center_px - box_h / 2)
                x2 = int(x_center_px + box_w / 2)
                y2 = int(y_center_px + box_h / 2)
                
                # Draw box
                cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img_rgb, f"Watermark", (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    return img_rgb

# Show random samples
SAMPLES_TO_SHOW = 12

images_folder = '/kaggle/working/dataset/images'
labels_folder = '/kaggle/working/dataset/labels'

all_imgs = [f for f in os.listdir(images_folder) if f.endswith('.jpg')]
num_samples = min(SAMPLES_TO_SHOW, len(all_imgs))
random_samples = random.sample(all_imgs, num_samples)

# Grid layout
if num_samples <= 6:
    rows, cols = 2, 3
elif num_samples <= 12:
    rows, cols = 3, 4
else:
    rows, cols = 4, 5

fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3))
axes = axes.flatten() if num_samples > 1 else [axes]

print(f"📊 Showing {num_samples} labeled samples:\n")

for idx, img_file in enumerate(random_samples):
    img_path = os.path.join(images_folder, img_file)
    label_path = os.path.join(labels_folder, img_file.replace('.jpg', '.txt'))
    
    img_with_boxes = draw_yolo_boxes(img_path, label_path)
    
    axes[idx].imshow(img_with_boxes)
    axes[idx].set_title(f"{img_file[:20]}...", fontsize=8)
    axes[idx].axis('off')

for idx in range(num_samples, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"✅ Green boxes show watermark regions")
print(f"💡 If boxes look wrong, adjust manual_regions in Step 4")

## Step 5: Split Dataset (Train/Val)

In [ ]:
# Create train/val split
os.makedirs('/kaggle/working/dataset/images/train', exist_ok=True)
os.makedirs('/kaggle/working/dataset/images/val', exist_ok=True)
os.makedirs('/kaggle/working/dataset/labels/train', exist_ok=True)
os.makedirs('/kaggle/working/dataset/labels/val', exist_ok=True)

# Get all images
all_images = [f for f in os.listdir('/kaggle/working/dataset/images') 
              if f.endswith('.jpg') and os.path.isfile(f'/kaggle/working/dataset/images/{f}')]

random.shuffle(all_images)

# 80/20 split
split_idx = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

print(f"📊 Splitting {len(all_images)} images...")

# Move train files
for img in train_images:
    src_img = f'/kaggle/working/dataset/images/{img}'
    dst_img = f'/kaggle/working/dataset/images/train/{img}'
    shutil.move(src_img, dst_img)
    
    label = img.replace('.jpg', '.txt')
    src_label = f'/kaggle/working/dataset/labels/{label}'
    dst_label = f'/kaggle/working/dataset/labels/train/{label}'
    if os.path.exists(src_label):
        shutil.move(src_label, dst_label)

# Move val files
for img in val_images:
    src_img = f'/kaggle/working/dataset/images/{img}'
    dst_img = f'/kaggle/working/dataset/images/val/{img}'
    shutil.move(src_img, dst_img)
    
    label = img.replace('.jpg', '.txt')
    src_label = f'/kaggle/working/dataset/labels/{label}'
    dst_label = f'/kaggle/working/dataset/labels/val/{label}'
    if os.path.exists(src_label):
        shutil.move(src_label, dst_label)

print(f"✅ Dataset split complete!")
print(f"   Train: {len(train_images)} images")
print(f"   Val: {len(val_images)} images")

## Step 6: Create data.yaml

In [ ]:
# Create YOLO config file
data_yaml = """path: /kaggle/working/dataset
train: images/train
val: images/val

names:
  0: watermark
"""

with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ data.yaml created")
print(f"📁 Config saved to: /kaggle/working/dataset/data.yaml")

## Step 7: Train YOLO Model 🚀

**⚠️ Make sure GPU is enabled!** (Settings → Accelerator → GPU T4 x2)

Training will take 2-4 hours depending on dataset size.

In [ ]:
from ultralytics import YOLO

# ========================================
# 🎯 TRAINING CONFIGURATION
# ========================================
EPOCHS = 200
BATCH_SIZE = 16  # Adjust based on GPU memory (8, 16, or 32)
IMAGE_SIZE = 640
PATIENCE = 30
model = YOLO('yolov8n.pt')  # Options: yolov8n.pt, yolov8s.pt, yolov8m.pt, yolov8l.pt, yolov8x.pt

print("🏋️ Starting YOLO training...")
print(f"⚙️ Epochs: {EPOCHS} (max), Patience: {PATIENCE}")
print(f"⚙️ Batch size: {BATCH_SIZE}, Image size: {IMAGE_SIZE}")
print(f"🎮 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print("\n⏰ This will take 2-4 hours...\n")

results = model.train(
    data='dataset/data.yaml',
    epochs=EPOCHS,  # Fewer epochs = faster
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    name='watermark_detector',
    device=0,  # Use GPU
    patience=PATIENCE,
    amp=False
)

print("\n✅ Training complete!")
print("📁 Model saved to: /kaggle/working/runs/detect/watermark_detector/weights/best.pt")
print("📊 Training plots saved to: /kaggle/working/runs/detect/watermark_detector/")

## Step 7.1: View Training Results 📊

Check if the model learned properly!

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

results_dir = '/kaggle/working/runs/detect/watermark_detector'

# Show training metrics
csv_path = f'{results_dir}/results.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    print("📊 Training Progress (Last 10 epochs):\n")
    print(df[['epoch', 'train/box_loss', 'train/cls_loss', 'metrics/mAP50(B)']].tail(10).to_string(index=False))
    
    final = df.iloc[-1]
    print(f"\n🎯 Final Metrics:")
    print(f"   mAP@50: {final['metrics/mAP50(B)']:.3f}")
    print(f"   mAP@50-95: {final['metrics/mAP50-95(B)']:.3f}")
    
    map50 = final['metrics/mAP50(B)']
    if map50 < 0.3:
        print("\n⚠️ LOW mAP - Model didn't learn well. Need more data or training.")
    elif map50 < 0.6:
        print("\n⚠️ MODERATE mAP - Model learned but could be better.")
    else:
        print("\n✅ GOOD mAP - Model learned successfully!")
else:
    print("❌ Results file not found")

# Show training curves
print("\n📈 Training Curves:")
curves = f'{results_dir}/results.png'
if os.path.exists(curves):
    display(IPImage(filename=curves, width=900))

# Show confusion matrix
print("\n📊 Confusion Matrix:")
cm = f'{results_dir}/confusion_matrix.png'
if os.path.exists(cm):
    display(IPImage(filename=cm, width=700))

## Step 8: Test Model on Validation Images

In [ ]:
from IPython.display import Image as IPImage, display

# Load trained model
model_path = '/kaggle/working/runs/detect/watermark_detector/weights/best.pt'
model = YOLO(model_path)

# Get validation images
val_images = [f for f in os.listdir('/kaggle/working/dataset/images/val') if f.endswith('.jpg')]

if not val_images:
    print("❌ No validation images found!")
else:
    print(f"📊 Testing on {len(val_images)} validation images\n")
    
    num_to_test = min(6, len(val_images))
    CONFIDENCE = 0.1  # Lower threshold to see all detections
    
    print(f"🔍 Confidence threshold: {CONFIDENCE}\n")
    
    # Test and display
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    total_detections = 0
    images_with_detections = 0
    
    for idx in range(num_to_test):
        img_path = f'/kaggle/working/dataset/images/val/{val_images[idx]}'
        
        # Run detection
        results = model(img_path, conf=CONFIDENCE, verbose=False)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        num_boxes = len(results[0].boxes)
        total_detections += num_boxes
        if num_boxes > 0:
            images_with_detections += 1
        
        axes[idx].imshow(annotated_rgb)
        axes[idx].set_title(f"{val_images[idx][:20]}...\n{num_boxes} detections", fontsize=9)
        axes[idx].axis('off')
        
        if num_boxes > 0:
            print(f"✅ {val_images[idx]}: {num_boxes} watermarks")
            for box in results[0].boxes:
                print(f"   - Confidence: {box.conf[0].cpu().numpy():.3f}")
        else:
            print(f"❌ {val_images[idx]}: No detections")
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Summary:")
    print(f"   Total detections: {total_detections}")
    print(f"   Detection rate: {images_with_detections}/{num_to_test}")
    
    if total_detections == 0:
        print("\n⚠️ NO DETECTIONS!")
        print("Try: Lower CONFIDENCE or add more training data")

## Step 9: Download Trained Model ⬇️

**Important:** The model is automatically saved to Kaggle's output directory!

### How to Download:

1. **Wait for notebook to finish** (all cells complete)
2. Click **💾 Save Version** (top right)
3. Choose **Save & Run All** → **Quick Save**
4. After completion, go to the **Output** tab (top of notebook)
5. Find `best.pt` in the output files
6. Click **⋮** (three dots) → **Download**

**File location in output:**
`runs/detect/watermark_detector/weights/best.pt`

In [ ]:
# Verify model file exists
model_path = '/kaggle/working/runs/detect/watermark_detector/weights/best.pt'

if os.path.exists(model_path):
    file_size = os.path.getsize(model_path) / (1024 * 1024)  # MB
    print(f"✅ Model trained successfully!")
    print(f"📁 Location: {model_path}")
    print(f"📦 Size: {file_size:.2f} MB")
    print("\n📥 To download:")
    print("1. Save this notebook (top right)")
    print("2. Go to Output tab")
    print("3. Download 'best.pt' from the output files")
    print("\n💡 Save to: VIDEO_EDITING_WATERMARK/models/best.pt")
else:
    print("❌ Model file not found!")
    print("Make sure training (Step 7) completed successfully")

---

# 🎉 Training Complete!

## What you got:
- ✅ Trained YOLO model (`best.pt`)
- ✅ Training metrics and plots
- ✅ Tested on validation images

## Next Steps:
1. **Download `best.pt`** from Output tab (after saving notebook)
2. Place in: `VIDEO_EDITING_WATERMARK/models/best.pt`
3. Update `video_editing.py`: Change `WATERMARK_MODE = "yolo"`
4. Install ultralytics locally: `pip install ultralytics`
5. Run your video processing!

## Kaggle Tips:
- 💾 Always **Save Version** before closing
- 📊 Check **Output** tab for all generated files
- ⏰ Free GPU limit: 30 hours/week
- 🔄 Can resume training from saved checkpoint

---